# 🎙️ Speech Emotion Recognition (SER)
### Using MFCC + Chroma + Mel Spectrogram  |  Architecture: LSTM → Dense

---
| | |
|---|---|
| **Dataset** | RAVDESS — 1440 `.wav` files, 24 actors |
| **Task** | Classify speech into 8 emotions |
| **Features** | MFCC (40) + Chroma (12) + Mel Spectrogram (128) — **full time-series, no mean** |
| **Model** | LSTM (last hidden state) → Dense layers → Softmax |
| **Emotions** | neutral · calm · happy · sad · angry · fear · disgust · surprised |

---
## ▶️ How to Run
1. Click **Runtime → Run all** in the menu above
2. When prompted, **upload** your `Audio_Speech_Actors_01-24.zip` file
3. Wait — the notebook will train and show all results automatically

---
## 📦 Cell 1 — Install Libraries

In [ ]:
!pip install -q librosa tensorflow scikit-learn matplotlib seaborn
print("✅ All libraries installed!")

---
## 📚 Cell 2 — Imports

In [ ]:
import os, zipfile, warnings
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

warnings.filterwarnings('ignore')
print(f"✅ TensorFlow {tf.__version__} | NumPy {np.__version__}")

---
## 📂 Cell 3 — Upload & Extract Dataset

In [ ]:
from google.colab import files

print("⬆️  Upload your RAVDESS zip file: Audio_Speech_Actors_01-24.zip")
uploaded = files.upload()

zip_path = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/ravdess')

actors = os.listdir('/content/ravdess')
print(f"✅ Extracted {len(actors)} actors: {sorted(actors)[:5]} ...")

---
## 🎭 Cell 4 — Emotion Labels

In [ ]:
EMOTION_MAP = {
    '01': 'neutral',  '02': 'calm',     '03': 'happy',    '04': 'sad',
    '05': 'angry',    '06': 'fear',     '07': 'disgust',  '08': 'surprised'
}

# Config
MAX_TIME_STEPS = 130
N_MFCC, N_CHROMA, N_MEL = 40, 12, 128
TOTAL_FEATURES = N_MFCC + N_CHROMA + N_MEL   # 180

print("Emotions:", list(EMOTION_MAP.values()))
print(f"Feature vector per time step: {TOTAL_FEATURES}  "
      f"(MFCC:{N_MFCC} + Chroma:{N_CHROMA} + Mel:{N_MEL})")

---
## 🔬 Cell 5 — Feature Extraction
> **No mean taken** — full time-series preserved → shape `(130, 180)` per file

In [ ]:
def extract_features(file_path):
    """Extract MFCC + Chroma + Mel as full time-series. No mean."""
    audio, sr = librosa.load(file_path, duration=3, offset=0.5)

    # MFCC: (40, T) → transpose → (T, 40)
    mfcc   = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC).T

    # Chroma: (12, T) → transpose → (T, 12)
    chroma = librosa.feature.chroma_stft(y=audio, sr=sr).T

    # Mel Spectrogram: (128, T) → dB scale → transpose → (T, 128)
    mel    = librosa.power_to_db(
                 librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=N_MEL),
                 ref=np.max
             ).T

    # Align time dimension
    T = min(mfcc.shape[0], chroma.shape[0], mel.shape[0])
    feat = np.concatenate([mfcc[:T], chroma[:T], mel[:T]], axis=1)  # (T, 180)

    # Pad or truncate to MAX_TIME_STEPS
    if feat.shape[0] < MAX_TIME_STEPS:
        feat = np.pad(feat, ((0, MAX_TIME_STEPS - feat.shape[0]), (0, 0)))
    else:
        feat = feat[:MAX_TIME_STEPS]

    return feat  # (130, 180)


# Quick test
for root, _, fs in os.walk('/content/ravdess'):
    for f in fs:
        if f.endswith('.wav'):
            test = extract_features(os.path.join(root, f))
            print(f"✅ Feature shape: {test.shape}  — expected ({MAX_TIME_STEPS}, {TOTAL_FEATURES})")
            break
    break

---
## 📊 Cell 6 — Build Dataset

In [ ]:
X, y, file_names = [], [], []

for root, _, fs in os.walk('/content/ravdess'):
    for f in sorted(fs):
        if not f.endswith('.wav'): continue
        code = f.split('-')[2]
        if code not in EMOTION_MAP: continue
        X.append(extract_features(os.path.join(root, f)))
        y.append(EMOTION_MAP[code])
        file_names.append(f)

X = np.array(X)   # (N, 130, 180)
y = np.array(y)

print(f"✅ Dataset: {X.shape[0]} files  |  Shape: {X.shape}")
print("\nSamples per emotion:")
for em in np.unique(y):
    print(f"  {em:12s}: {np.sum(y==em)}")

---
## ⚙️ Cell 7 — Preprocessing

In [ ]:
# Normalize (z-score across all samples & time steps, per feature)
N, T, F   = X.shape
X_flat    = X.reshape(-1, F)
X_mean    = X_flat.mean(axis=0)
X_std     = X_flat.std(axis=0) + 1e-8
X_norm    = (X - X_mean) / X_std          # (N, 130, 180)

# Encode labels
le          = LabelEncoder()
y_enc       = le.fit_transform(y)
y_cat       = to_categorical(y_enc)
NUM_CLASSES = y_cat.shape[1]

# Train / Test split (80/20, stratified)
X_train, X_test, y_train, y_test, y_train_enc, y_test_enc = train_test_split(
    X_norm, y_cat, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

print(f"✅ Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"   Classes ({NUM_CLASSES}): {le.classes_}")

---
## 🧠 Cell 8 — Model Architecture

```
Input  (130, 180)
  │
  ▼
LSTM (128 units, return_sequences=False)
  │  └─ outputs only LAST hidden state → (128,)  ← single 1D vector
  ▼
Dropout (0.4)
  ▼
Dense (256, ReLU) + BatchNorm + Dropout (0.3)
  ▼
Dense (128, ReLU) + BatchNorm + Dropout (0.3)
  ▼
Dense (8, Softmax)  →  8 emotion classes
```

In [ ]:
def build_model(input_shape, num_classes):
    inp = Input(shape=input_shape, name='Input')           # (130, 180)

    # ── LSTM: full sequence in → last hidden state out
    x = LSTM(128, return_sequences=False, name='LSTM')(inp) # (128,)
    x = Dropout(0.4,  name='Drop_LSTM')(x)

    # ── Dense layers after LSTM
    x = Dense(256, activation='relu', name='Dense_1')(x)   # (256,)
    x = BatchNormalization(name='BN_1')(x)
    x = Dropout(0.3, name='Drop_1')(x)

    x = Dense(128, activation='relu', name='Dense_2')(x)   # (128,)
    x = BatchNormalization(name='BN_2')(x)
    x = Dropout(0.3, name='Drop_2')(x)

    out = Dense(num_classes, activation='softmax', name='Output')(x)

    return Model(inp, out, name='LSTM_Dense_SER')


model = build_model((MAX_TIME_STEPS, TOTAL_FEATURES), NUM_CLASSES)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

---
## 🏋️ Cell 9 — Train Model

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=15,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=5, min_lr=1e-6, verbose=1)
]

history = model.fit(
    X_train, y_train,
    epochs=80, batch_size=32,
    validation_data=(X_test, y_test),
    callbacks=callbacks, verbose=1
)

---
## 📈 Cell 10 — Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training History', fontsize=16, fontweight='bold')

ax1.plot(history.history['accuracy'],     label='Train', color='steelblue', lw=2)
ax1.plot(history.history['val_accuracy'], label='Val',   color='tomato',    lw=2)
ax1.set_title('Accuracy vs Epochs'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(history.history['loss'],     label='Train', color='steelblue', lw=2)
ax2.plot(history.history['val_loss'], label='Val',   color='tomato',    lw=2)
ax2.set_title('Loss vs Epochs'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Best Val Accuracy: {max(history.history['val_accuracy'])*100:.2f}%")

---
## 🧪 Cell 11 — Evaluation (Accuracy, Precision, Recall, F1)

In [ ]:
y_pred        = model.predict(X_test, verbose=0)
y_pred_cls    = np.argmax(y_pred,  axis=1)
y_true_cls    = np.argmax(y_test,  axis=1)

loss, acc = model.evaluate(X_test, y_test, verbose=0)

print("=" * 55)
print(f"  Test Accuracy : {acc*100:.2f}%")
print(f"  Test Loss     : {loss:.4f}")
print("=" * 55)
print()
print(classification_report(y_true_cls, y_pred_cls, target_names=le.classes_))

---
## 🟦 Cell 12 — Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true_cls, y_pred_cls)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix — Speech Emotion Recognition', fontsize=14, fontweight='bold')
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual',    fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🔍 Cell 13 — Overfitting / Underfitting Analysis

In [ ]:
train_acc = history.history['accuracy'][-1]
val_acc   = history.history['val_accuracy'][-1]
gap       = train_acc - val_acc

print(f"Train Accuracy : {train_acc*100:.2f}%")
print(f"Val   Accuracy : {val_acc*100:.2f}%")
print(f"Gap            : {gap*100:.2f}%")
print()

if gap > 0.15:
    print("⚠️  OVERFITTING — gap > 15%")
    print("   Fix: increase dropout · add L2 reg · data augmentation")
elif val_acc < 0.50:
    print("⚠️  UNDERFITTING — val accuracy < 50%")
    print("   Fix: deeper model · more epochs · check feature extraction")
else:
    print("✅ Model is generalising well!")

---
## 🔄 Cell 14 — Multiple Training Versions (Hyperparameter Study)

In [ ]:
def run_experiment(cfg, name):
    print(f"  Training {name} ...", end=' ')
    inp = Input(shape=(MAX_TIME_STEPS, TOTAL_FEATURES))
    x   = LSTM(cfg['lstm'], return_sequences=False)(inp)
    x   = Dropout(cfg['drop'])(x)
    x   = Dense(cfg['dense'], activation='relu')(x)
    x   = BatchNormalization()(x)
    x   = Dropout(cfg['drop'])(x)
    x   = Dense(cfg['dense']//2, activation='relu')(x)
    x   = BatchNormalization()(x)
    x   = Dropout(cfg['drop'])(x)
    out = Dense(NUM_CLASSES, activation='softmax')(x)
    m   = Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(cfg['lr']),
              loss='categorical_crossentropy', metrics=['accuracy'])
    m.fit(X_train, y_train, epochs=50, batch_size=cfg['bs'],
          validation_data=(X_test, y_test),
          callbacks=[EarlyStopping(monitor='val_accuracy', patience=10,
                                   restore_best_weights=True)],
          verbose=0)
    _, v = m.evaluate(X_test, y_test, verbose=0)
    print(f"Val Acc = {v*100:.2f}%")
    return v


EXPERIMENTS = [
    # Name,         lstm,  dense, drop,  lr,      bs
    ('Version 1 — Small  (LR=0.001, BS=32, LSTM=64)',  {'lstm':64,  'dense':128, 'drop':0.3, 'lr':0.001,  'bs':32}),
    ('Version 2 — Medium (LR=0.001, BS=32, LSTM=128)', {'lstm':128, 'dense':256, 'drop':0.4, 'lr':0.001,  'bs':32}),
    ('Version 3 — Large  (LR=5e-4,  BS=64, LSTM=256)', {'lstm':256, 'dense':512, 'drop':0.5, 'lr':0.0005, 'bs':64}),
]

rows = []
print("Running experiments...")
for name, cfg in EXPERIMENTS:
    acc = run_experiment(cfg, name)
    rows.append({'Version': name, 'LSTM Units': cfg['lstm'], 'Dense Units': cfg['dense'],
                 'Dropout': cfg['drop'], 'LR': cfg['lr'], 'Batch': cfg['bs'],
                 'Val Accuracy': f"{acc*100:.2f}%"})

df = pd.DataFrame(rows)
print("\n" + "="*80)
print("EXPERIMENT RESULTS")
print("="*80)
print(df.to_string(index=False))

best = df.loc[df['Val Accuracy'].str.rstrip('%').astype(float).idxmax()]
print(f"\n🏆 Best Model: {best['Version']}  →  {best['Val Accuracy']}")
df.to_csv('experiment_results.csv', index=False)

---
## 💾 Cell 15 — Save Full Predictions to CSV

In [ ]:
all_preds = model.predict(X_norm, verbose=0)
all_pred_cls = np.argmax(all_preds, axis=1)

results = pd.DataFrame({
    'File Name':         file_names,
    'Actual Emotion':    le.inverse_transform(y_enc),
    'Predicted Emotion': le.inverse_transform(all_pred_cls)
})
results.to_csv('final_predictions.csv', index=False)

correct = (results['Actual Emotion'] == results['Predicted Emotion']).sum()
print(f"✅ Saved {len(results)} predictions → final_predictions.csv")
print(f"   Overall Accuracy: {correct}/{len(results)} = {correct/len(results)*100:.2f}%")
results.head(10)

---
## 📋 Cell 16 — Project Summary

In [ ]:
final_val_acc = max(history.history['val_accuracy'])

summary = {
    'Problem Statement':  'Speech Emotion Recognition (8-class classification)',
    'Dataset':            'RAVDESS — 1440 .wav files, 24 actors',
    'Input':              '.wav audio files (3 sec clips)',
    'Output':             '8 emotions: neutral, calm, happy, sad, angry, fear, disgust, surprised',
    'Features':           'MFCC(40) + Chroma(12) + Mel Spectrogram(128) = 180 features/step',
    'Sequence':           'Full time-series (130 steps) — NO mean aggregation',
    'Architecture':       'LSTM(128) → Dense(256) → Dense(128) → Softmax(8)',
    'Loss Function':      'Categorical Cross-Entropy',
    'Optimizer':          'Adam',
    'Regularization':     'BatchNorm + Dropout + EarlyStopping + ReduceLROnPlateau',
    'Best Val Accuracy':  f'{final_val_acc*100:.2f}%',
}

print("\n" + "="*60)
print("        PROJECT SUMMARY — Speech Emotion Recognition")
print("="*60)
for k, v in summary.items():
    print(f"  {k:<22}: {v}")
print("="*60)